In [ ]:
# ============================================================
# CELDA 0: Imports, estilo y base AIMA
# ============================================================
# --- Librerías ---
import sys, time, tracemalloc, heapq
import statistics as st
from collections import deque

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Rectangle
from IPython.display import HTML

# --- Config Matplotlib ---
plt.rcParams["animation.embed_limit"] = 200      

# --- AIMA-Python ---
sys.path.insert(0, "aima-python")                
try:
    from aima.search import (Problem, breadth_first_graph_search,
                             depth_first_graph_search, astar_search)
except ImportError:                             
    from search import (Problem, breadth_first_graph_search,
                        depth_first_graph_search, astar_search)

# --- Colores ---
BG, BLANCO, TEXTO = "#1E1E1E", "#F2F2F2", "#B5B5B5"
C_LIMPIO, C_INF, C_AISL = "#A8C690", "#C25A6E", "#7FA6C4"     
C_PENDIENTE, C_EXPLORADO = "#E5CC8A", "#4A4A4A"               
print("OK: imports y AIMA listos")

In [ ]:
# ============================================================
# CELDA 1: Formalización (hereda de aima.search.Problem)
# ============================================================
# --- Red de cuatro nodos ---
NOMBRES  = ["Servidor", "Subestación", "Cámara 1", "Cámara 2"]
CRITICOS = (0, 1)                                    
ARISTAS  = [(0, 1), (2, 3), (0, 2), (1, 3)]          
VECINOS  = {i: tuple(sorted({b for a, b in ARISTAS if a == i} | {a for a, b in ARISTAS if b == i}))
            for i in range(len(NOMBRES))}

# --- Estados y costos ---
LIMPIO, INFECTADO, AISLADO = 0, 1, 2
COSTO_ACCION  = {"esperar": 1, "aislar": 2, "restablecer": 3}
PESO_SERVICIO = (3, 3, 1, 1)                         

class WatchdogProblem(Problem):
    """Espacio de estados de Watchdog (estado = tupla de cuatro valores)."""

    def actions(self, s):
        """Acciones disponibles en el estado s."""
        acts = []
        for i, v in enumerate(s):
            if v == INFECTADO:                       # un nodo infectado se puede contener o limpiar
                acts += [("aislar", i), ("restablecer", i)]
            elif v == AISLADO:                       # un nodo aislado se puede reconectar (queda limpio)
                acts.append(("restablecer", i))
        acts.append(("esperar",))                    # los nodos limpios no admiten acciones propias
        return acts

    def result(self, s, a):            
        """Aplica la acción y propaga la infección a los vecinos limpios."""            
        est = list(s)
        if a[0] == "aislar":
            est[a[1]] = AISLADO
        elif a[0] == "restablecer":
            est[a[1]] = LIMPIO
        infectados = [i for i, v in enumerate(est) if v == INFECTADO]
        for i in infectados:                         # propagación simultánea
            for j in VECINOS[i]:
                if est[j] == LIMPIO:
                    est[j] = INFECTADO
        return tuple(est)

    def goal_test(self, s):
        """Sin nodos infectados y con servidor y subestación activos."""
        return INFECTADO not in s and all(s[i] != AISLADO for i in CRITICOS)

    def path_cost(self, c, s1, a, s2):
        """Costo de la acción más el peso de los servicios aislados."""
        servicios_caidos = sum(PESO_SERVICIO[i] for i, v in enumerate(s2) if v == AISLADO)
        return c + COSTO_ACCION[a[0]] + servicios_caidos

    def heuristica(self, s):
        """Heurística admisible, tres unidades por cada nodo infectado."""
        return 3 * s.count(INFECTADO)

    def h(self, node):
        """Versión de la heurística que espera AIMA (recibe un Node)."""      
        return self.heuristica(node.state)

# --- Instancia inicial ---
ESTADO_INICIAL = (LIMPIO, LIMPIO, INFECTADO, INFECTADO)
problema = WatchdogProblem(ESTADO_INICIAL)

print("Estado inicial :", ESTADO_INICIAL, "| hash OK:", hash(ESTADO_INICIAL) is not None)
print("Acciones       :", problema.actions(ESTADO_INICIAL))
print("Ejemplo result :", problema.result(ESTADO_INICIAL, ("restablecer", 2)))
print("|S| total      :", 3 ** len(NOMBRES), "estados posibles")
print("Meta           : ningún nodo infectado y Servidor/Subestación sin aislar")

In [ ]:
# ============================================================
# CELDA 2: Algoritmos de búsqueda en grafo (BFS, DFS, A*)
# ============================================================
def reconstruir(padre, accion, estado):
    """Reconstruye el camino desde el estado inicial hasta el estado dado."""
    estados, acciones = [], []
    while estado is not None:
        estados.append(estado)
        acciones.append(accion[estado])
        estado = padre[estado]
    return estados[::-1], acciones[::-1][1:]

def busqueda(problema, estrategia, w=1.0, capturar=True):
    """Búsqueda en grafo con estrategia bfs, dfs o astar, donde f = g + w*h."""
    
    # --- Estructuras de la búsqueda ---
    ini = problema.initial
    padre, accion = {ini: None}, {ini: None}
    prof, g = {ini: 0}, {ini: 0}
    orden = [ini]                                    
    cerrados, en_frontera = set(), {ini}             
    generados, contador, n_exp, frames = 1, 0, 0, []

    # --- Frontera según la estrategia ---
    if estrategia == "bfs":
        frontera = deque([ini])                      # cola FIFO
    elif estrategia == "dfs":
        frontera = [ini]                             # pila LIFO
    else:
        frontera = [(w * problema.heuristica(ini), contador, ini)]   # cola de prioridad por f

    def snap():
        return {"padre": dict(padre), "prof": dict(prof), "orden": list(orden)}

    # --- Bucle principal ---
    while frontera:
        if estrategia == "bfs":
            actual = frontera.popleft()
        elif estrategia == "dfs":
            actual = frontera.pop()
        else:
            _, _, actual = heapq.heappop(frontera)
        if actual in cerrados:                       # entrada obsoleta de la cola
            continue
        en_frontera.discard(actual)
        cerrados.add(actual)
        n_exp += 1
        meta = problema.goal_test(actual)

         #--- Fotograma y prueba de meta ---
        if capturar:
            frames.append({"estado": actual, "expandidos": set(cerrados),
                           "frontera": set(en_frontera), "snap": snap()})
        if meta:
            estados, acciones = reconstruir(padre, accion, actual)
            return {"camino": estados, "acciones": acciones, "costo": g[actual],
                    "pasos": len(acciones), "expandidos": n_exp, "generados": generados,
                    "prof_max": max(prof.values()), "frames": frames}

        # --- Expansión del nodo ---
        for a in problema.actions(actual):
            hijo = problema.result(actual, a)
            if hijo in cerrados:
                continue
            c = problema.path_cost(g[actual], actual, a, hijo)
            # A* se reinserta el estado si aparece un camino más barato
            if estrategia == "astar":
                if c < g.get(hijo, float("inf")):    
                    if hijo not in prof:
                        orden.append(hijo)
                    padre[hijo], accion[hijo], g[hijo] = actual, a, c
                    prof[hijo] = prof[actual] + 1
                    contador += 1
                    generados += 1
                    en_frontera.add(hijo)
                    heapq.heappush(frontera, (c + w * problema.heuristica(hijo), contador, hijo))
            # BFS y DFS cada estado entra una sola vez a la frontera
            elif hijo not in en_frontera:            
                orden.append(hijo)
                padre[hijo], accion[hijo], g[hijo] = actual, a, c
                prof[hijo] = prof[actual] + 1
                generados += 1
                en_frontera.add(hijo)
                frontera.append(hijo)

    return {"camino": None, "acciones": [], "costo": None, "pasos": 0, "expandidos": n_exp,
            "generados": generados, "prof_max": max(prof.values()), "frames": frames}

def texto_accion(a):
    return "esperar" if a[0] == "esperar" else f"{a[0]} {NOMBRES[a[1]]}"

r = busqueda(problema, "astar", capturar=False)
print("Prueba A*  -> plan:", [texto_accion(a) for a in r["acciones"]])
print("             costo:", r["costo"], "| expandidos:", r["expandidos"])

In [ ]:
# ============================================================
# CELDA 3: Benchmark (nodos expandidos, tiempo, costo) + validaciones
# ============================================================
def medir(nombre, problema, estrategia, w=1.0, rep=5):
    """Mide el tiempo (mediana) y la memoria pico de un algoritmo."""
    tiempos = []
    for _ in range(rep):
        t0 = time.perf_counter()
        r = busqueda(problema, estrategia, w, capturar=False)
        tiempos.append((time.perf_counter() - t0) * 1000)
    tracemalloc.start()
    busqueda(problema, estrategia, w, capturar=False)
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    r.update(nombre=nombre, tiempo_ms=st.median(tiempos), memoria_kb=pico / 1024)
    return r

# --- Benchmark ---
CONFIG = [("BFS", "bfs", 1.0), ("DFS", "dfs", 1.0),
          ("A* (h admisible)", "astar", 1.0)]
resultados = [medir(n, problema, e, w) for n, e, w in CONFIG]
mio = {r["nombre"]: r for r in resultados}
C_OPT = mio["A* (h admisible)"]["costo"]

# --- Tabla de resultados ---
tabla = pd.DataFrame([{
    "Algoritmo": r["nombre"], "Pasos": r["pasos"], "Costo": r["costo"],
    "¿Óptimo?": "Sí" if r["costo"] == C_OPT else "No",
    "Expandidos": r["expandidos"], "Generados": r["generados"], "Prof. máx": r["prof_max"],
    "Tiempo (ms)": round(r["tiempo_ms"], 2), "Memoria (KB)": round(r["memoria_kb"], 1)}
    for r in resultados]).set_index("Algoritmo")
display(tabla)

# --- Validación con AIMA --
aima_bfs = breadth_first_graph_search(WatchdogProblem(ESTADO_INICIAL))
aima_dfs = depth_first_graph_search(WatchdogProblem(ESTADO_INICIAL))
aima_ast = astar_search(WatchdogProblem(ESTADO_INICIAL))
print("\nValidación contra AIMA:")
print(f"  BFS  pasos   propio={mio['BFS']['pasos']}  AIMA={len(aima_bfs.solution())}")
print(f"  A*   costo   propio={C_OPT}  AIMA={aima_ast.path_cost}")
print(f"  DFS  resuelve: propio={mio['DFS']['camino'] is not None}  AIMA={aima_dfs is not None}")

# --- Validación de la heurística ---
def alcanzables(ini):
    """Estados alcanzables desde ini (recorrido en anchura)."""
    vistos, cola = {ini}, deque([ini])
    while cola:
        s = cola.popleft()
        for a in problema.actions(s):
            t = problema.result(s, a)
            if t not in vistos:
                vistos.add(t); cola.append(t)
    return vistos

TODOS = alcanzables(ESTADO_INICIAL)
resueltos = admisibles = 0
for s in TODOS:
    # costo óptimo real desde s, calculado sin heurística
    real = busqueda(WatchdogProblem(s), "astar", w=0.0, capturar=False)["costo"]
    if real is not None:
        resueltos += 1
        admisibles += problema.heuristica(s) <= real
print(f"  h admisible: h(s) <= costo óptimo real en {admisibles}/{resueltos} estados alcanzables")

In [ ]:
# ============================================================
# CELDA 4: Parámetros b, d, m, C*, ε y factor de ramificación efectivo
# ============================================================
def factor_efectivo(N, d):
    """Resuelve N + 1 = 1 + b + ... + b^d para b por bisección."""
    if d == 0:
        return 0.0
    lo, hi = 1.0001, 60.0
    for _ in range(80):
        mid = (lo + hi) / 2
        if sum(mid ** k for k in range(d + 1)) < N + 1:
            lo = mid
        else:
            hi = mid
    return round((lo + hi) / 2, 3)

# --- Parámetros del análisis ---
b = max(len(problema.actions(s)) for s in TODOS)     # máx. acciones en un estado
d = mio["BFS"]["pasos"]                              # profundidad de la solución menos profunda
m = mio["DFS"]["prof_max"]                           # profundidad máxima explorada
eps = min(COSTO_ACCION.values())                     # costo mínimo de un paso
print(f"b = {b} | d = {d} | m = {m} | C* = {C_OPT} | ε = {eps} | estados alcanzables = {len(TODOS)}")

# --- Big-O teórico ---
print("\nTeoría (Russell & Norvig):")
print("  BFS   tiempo O(b^d)                 espacio O(b^d)")
print("  DFS   tiempo O(b^m)                 espacio O(b·m)")
print("  A*    tiempo O(b^d) peor caso       espacio O(b^d)   (con h: b* << b)")

# --- Factor de ramificación efectivo ---
print("\nFactor de ramificación efectivo medido (menor = búsqueda más dirigida):")
for r in resultados:
    if r["nombre"] != "DFS":                         # b* se calcula solo para BFS y A*
        print(f"  {r['nombre']:<38} b* = {factor_efectivo(r['generados'], max(r['pasos'], 1))}")

In [ ]:
# ============================================================
# CELDA 5: Dibujo de la red, del árbol y construcción del video
# ============================================================
# --- Configuración de la figura ---
FIG_W, FIG_H = 14, 7
COL_ESTADO = {LIMPIO: C_LIMPIO, INFECTADO: C_INF, AISLADO: C_AISL}
FUENTE = "DejaVu Sans Mono"

def _prep(ax):
    """Prepara el eje con la pulgada como unidad de datos."""
    ax.clear(); ax.set_facecolor(BG); ax.axis("off")
    bb = ax.get_position(original=True)              
    W, H = bb.width * FIG_W, bb.height * FIG_H
    ax.set_xlim(0, W); ax.set_ylim(0, H)             
    return W, H

def _leyenda(ax, W, items, y=0.25):
    """Dibuja una fila de cuadros de color con su etiqueta."""
    paso = 2.0
    x0 = W / 2 - paso * (len(items) - 1) / 2 - 0.5
    for k, (col, txt) in enumerate(items):
        x = x0 + k * paso
        ax.add_patch(Rectangle((x - 0.12, y - 0.12), 0.24, 0.24, fc=col, ec="none"))
        ax.text(x + 0.25, y, txt, color=BLANCO, fontsize=12, family=FUENTE, va="center")

def dibujar_red(ax, estado, marcado=None, titulo="Red"):
    """Dibuja la red con el color de cada estado."""
    W, H = _prep(ax)
    # --- Posiciones de los nodos ---
    pos = {0: (W * 0.28, H * 0.66), 1: (W * 0.72, H * 0.66),
           2: (W * 0.28, H * 0.30), 3: (W * 0.72, H * 0.30)}
    # --- Conexiones ---
    for a, b_ in ARISTAS:                            
        va, vb = estado[a], estado[b_]
        if AISLADO in (va, vb):   col, ls = C_AISL, (0, (1, 2))        
        elif INFECTADO in (va, vb) and (LIMPIO in (va, vb)):
            col, ls = C_INF, "-"                                     
        else:                     col, ls = "#5A5A5A", "-"
        ax.plot([pos[a][0], pos[b_][0]], [pos[a][1], pos[b_][1]], ls=ls, color=col, lw=5, zorder=1)
    # --- Nodos ---
    for i, (x, y) in pos.items():
        # los nodos críticos se dibujan más grandes
        lado = 1.15 if i in CRITICOS else 0.8                          
        ax.add_patch(Rectangle((x - lado / 2, y - lado / 2), lado, lado,
                               fc=COL_ESTADO[estado[i]], ec="none", zorder=3))
        # marco blanco sobre el nodo en que actúa el agente
        if marcado == i:                                               
            ax.add_patch(Rectangle((x - lado / 2 - 0.1, y - lado / 2 - 0.1), lado + 0.2, lado + 0.2,
                                   fc="none", ec=BLANCO, lw=3, zorder=4))
        if i in CRITICOS:
            ax.text(x, y + lado / 2 + 0.2, NOMBRES[i], color=BLANCO, fontsize=13, ha="center", va="bottom")
            ax.text(x, y + lado / 2 + 0.55, "crítico", color=TEXTO, fontsize=10, ha="center", va="bottom")
        else:
            ax.text(x, y - lado / 2 - 0.2, NOMBRES[i], color=BLANCO, fontsize=13, ha="center", va="top")
    _leyenda(ax, W, [(C_LIMPIO, "limpio"), (C_INF, "infectado"), (C_AISL, "aislado")])
    ax.set_title(titulo, color=BLANCO, fontsize=13, fontweight="bold", pad=8)

def _glifo(ax, x, y, estado, g, plate):
    """Dibuja un estado como una mini-red de 2x2 bloques."""
    ax.add_patch(Rectangle((x - g * 0.62, y - g * 0.62), g * 1.24, g * 1.24, fc=plate, ec="none", zorder=3))
    c = g * 0.46
    for i, (dx, dy) in enumerate([(-1, 1), (1, 1), (-1, -1), (1, -1)]):
        cx, cy = x + dx * c * 0.55, y + dy * c * 0.55
        ax.add_patch(Rectangle((cx - c / 2, cy - c / 2), c, c, fc=COL_ESTADO[estado[i]], ec="none", zorder=4))

def dibujar_arbol(ax, snap, expandidos, frontera, camino, actual, titulo):
    """Dibuja el árbol de búsqueda, con cada nodo como una mini-red."""
    W, H = _prep(ax)
    padre, prof, orden = snap["padre"], snap["prof"], snap["orden"]
    # --- Capas por profundidad ---
    capas = {}
    for n in orden:
        capas.setdefault(prof[n], []).append(n)
    maxp, maxcapa = max(capas), max(len(v) for v in capas.values())
    # --- Tamaño y posición de los nodos ---
    vpaso = (H - 1.6) / max(maxp, 1)
    g = min(0.34, 0.55 * W / maxcapa, 0.5 * vpaso)                     
    pos = {n: ((i + 0.5) * W / len(nodos), H - 0.5 - d * vpaso)
           for d, nodos in capas.items() for i, n in enumerate(nodos)}
    # --- Aristas ---
    sol = set(zip(camino[:-1], camino[1:])) if camino and len(camino) > 1 else set()
    for n, (x, y) in pos.items():                                    
        p = padre[n]
        if p is not None:
            es = (p, n) in sol
            ax.plot([pos[p][0], x], [pos[p][1], y], "-", color=BLANCO if es else "#5A5A5A",
                    lw=4 if es else 1, zorder=2 if es else 1)
    # --- Nodos ---
    plan = set(camino or [])
    for n, (x, y) in pos.items():                                      
        if n == actual or n in plan:  plate = BLANCO
        elif n in frontera:           plate = C_PENDIENTE
        else:                         plate = C_EXPLORADO
        _glifo(ax, x, y, n, g, plate)
    _leyenda(ax, W, [(C_PENDIENTE, "pendiente"), (C_EXPLORADO, "explorado"), (BLANCO, "actual / plan")])
    ax.set_title(titulo, color=BLANCO, fontsize=13, fontweight="bold", pad=8)

def armar_frames(res, max_frames):
    # --- Fotogramas de la exploración ---
    fr = res["frames"]
    if len(fr) > max_frames:
        paso = len(fr) // max_frames
        idx = list(range(0, len(fr), paso))
        if idx[-1] != len(fr) - 1:
            idx.append(len(fr) - 1)
        fr = [fr[i] for i in idx]
    final = res["frames"][-1]
    out = [{"red": ESTADO_INICIAL, "marcado": None, "titulo_red": "Red (problema)",
            "expandidos": f["expandidos"], "frontera": f["frontera"], "snap": f["snap"],
            "camino": None, "actual": f["estado"], "fase": "busca"} for f in fr]
    # --- Fotogramas del plan ---
    camino, acc, g = res["camino"], res["acciones"], 0
    for k, est in enumerate(camino):
        if k > 0:
            g = problema.path_cost(g, camino[k - 1], acc[k - 1], est)
        a = acc[k - 1] if k > 0 else None
        marcado = a[1] if (a and a[0] != "esperar") else None
        t = "Plan - estado inicial" if k == 0 else f"Plan {k}/{len(acc)}: {texto_accion(a)}  (costo {g})" + ("  ✓ meta" if k == len(acc) else "")
        # cada paso se muestra en dos fotogramas
        for _ in range(2):                         
            out.append({"red": est, "marcado": marcado, "titulo_red": t,
                        "expandidos": final["expandidos"], "frontera": final["frontera"],
                        "snap": final["snap"], "camino": camino[:k + 1], "actual": est, "fase": "plan"})
    return out

def construir_animacion(nombre, estrategia, w=1.0, max_frames=100):
    """Crea la figura y la función que dibuja cada fotograma."""
    res = busqueda(problema, estrategia, w)
    frames = armar_frames(res, max_frames)
    # --- Figura y ejes ---
    fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor=BG, dpi=80)
    ax_red = fig.add_axes([0.03, 0.06, 0.40, 0.84])
    ax_arb = fig.add_axes([0.46, 0.06, 0.52, 0.84])

    def update(i):
        f = frames[i]
        dibujar_red(ax_red, f["red"], f["marcado"], f["titulo_red"])
        t = (f"{nombre} - Exp:{len(f['expandidos'])} Fron:{len(f['frontera'])}" if f["fase"] == "busca"
             else f"{nombre} - plan encontrado (costo {res['costo']})")
        dibujar_arbol(ax_arb, f["snap"], f["expandidos"], f["frontera"], f["camino"], f["actual"], t)
    return fig, update, len(frames), res

def simular(nombre, estrategia, w=1.0, fps=2, max_frames=100, guardar_gif=False):
    print(f"▶ Simulando {nombre}…")
    fig, update, total, res = construir_animacion(nombre, estrategia, w, max_frames)
    print(f"  plan: {res['pasos']} pasos | costo {res['costo']} | expandidos {res['expandidos']} | fotogramas {total}")
    anim = animation.FuncAnimation(fig, update, frames=total, interval=1000 // fps, repeat=False)
    plt.close(fig)
    if guardar_gif:
        ruta = "watchdog_" + estrategia + ("_w%g" % w if estrategia == "astar" else "") + ".gif"
        anim.save(ruta, writer="pillow", fps=max(fps, 5))
        print("  GIF guardado:", ruta)
    return HTML(anim.to_jshtml())

print("OK: visualización lista")

In [ ]:
# CELDA 6: BFS
simular("BFS", "bfs", fps=2, max_frames=100)

In [ ]:
# CELDA 7: DFS
simular("DFS", "dfs", fps=2, max_frames=100)

In [ ]:
# CELDA 8: A* admisible
simular("A*", "astar", w=1.0, fps=2, max_frames=100)

In [ ]:
# ============================================================
# CELDA 9: Medición de rendimiento
# ============================================================
# --- Algoritmos a medir ---
ALGORITMOS = [("BFS", "bfs", 1.0), ("DFS", "dfs", 1.0),
              ("A*", "astar", 1.0)]

def medir_rendimiento(nombre, estrategia, w=1.0, repeticiones=5):
    """Mide tiempo (media y desviación) y memoria pico de un algoritmo."""
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        r = busqueda(problema, estrategia, w, capturar=False)
        tiempos.append((time.perf_counter() - t0) * 1000)
    tracemalloc.start()
    busqueda(problema, estrategia, w, capturar=False)
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {"nombre": nombre, "pasos": r["pasos"], "costo": r["costo"],
            "expandidos": r["expandidos"], "generados": r["generados"],
            "prof_max": r["prof_max"], "camino": r["camino"],
            "tiempo_ms": st.mean(tiempos), "tiempo_std": st.pstdev(tiempos),
            "memoria_kb": pico / 1024}

# --- Mediciones ---
print("Midiendo rendimiento (5 repeticiones)...\n")
bench = []
for nombre, estrategia, w in ALGORITMOS:
    print(f"  Midiendo {nombre}...", end=" ", flush=True)
    bench.append(medir_rendimiento(nombre, estrategia, w))
    print(f"{bench[-1]['tiempo_ms']:.2f} ms")
print("\nMediciones completadas")

In [ ]:
# ============================================================
# CELDA 10: Tabla comparativa + Big-O + análisis + gráficas
# ============================================================
mejor_costo = min(r["costo"] for r in bench)

# --- Tabla comparativa ---
print()
print("=" * 98)
print(" COMPARACION DE RENDIMIENTO - WATCHDOG (red de 4 nodos)".center(98))
print(f" Estado inicial {ESTADO_INICIAL}  (Servidor, Subestación, Cámara 1, Cámara 2)".center(98))
print(f" b={b}  d={d}  m={m}  C*={mejor_costo}  ε={eps}".center(98))
print("=" * 98)
print(f" {'Algoritmo':<10} {'Pasos':<7} {'Costo':<7} {'Expand':<8} {'Gener.':<8} "
      f"{'Tiempo(ms)':<12} {'±σ':<8} {'Memoria(KB)':<12} {'b*':<6}")
print("-" * 98)
for r in bench:
    bstar = "—" if r["nombre"] == "DFS" else factor_efectivo(r["generados"], max(r["pasos"], 1))
    print(f" {r['nombre']:<10} {r['pasos']:<7} {r['costo']:<7} {r['expandidos']:<8} {r['generados']:<8} "
          f"{r['tiempo_ms']:<12.2f} {r['tiempo_std']:<8.3f} {r['memoria_kb']:<12.1f} {bstar:<6}")
print("=" * 98)

# --- Big-O teórico ---
print("\nPredicción teórica Big-O (Russell & Norvig, Tabla 3.1):\n")
print(f" {'Algoritmo':<10} {'Frontera':<20} {'Completo':<12} {'Óptimo':<16} {'Tiempo':<14} {'Espacio':<10}")
print("-" * 98)
big_o = {
    "BFS":      ("Cola FIFO",        "Sí",          "No (con pesos)", "O(b^d)", "O(b^d)"),
    "DFS":      ("Pila LIFO",        "Sí (finito)", "No",             "O(b^m)", "O(b·m)"),
    "A*":       ("Cola prioridad f", "Sí",          "Sí (h admis.)",  "O(b^d)", "O(b^d)"),
}
for r in bench:
    fr, comp, opt, t, e = big_o[r["nombre"]]
    print(f" {r['nombre']:<10} {fr:<20} {comp:<12} {opt:<16} {t:<14} {e:<10}")
print("-" * 98)
print(f" Con b={b}, d={d}, m={m}:  b^d = {b**d:,}   b^m = {b**m:,}   (cotas de peor caso)")

# --- Análisis ---
optimos = [r["nombre"] for r in bench if r["costo"] == mejor_costo]
mas_rapido = min(bench, key=lambda r: r["tiempo_ms"])
menos_exp = min(bench, key=lambda r: r["expandidos"])
menos_mem = min(bench, key=lambda r: r["memoria_kb"])
por = {r["nombre"]: r for r in bench}

print("\nAnálisis del experimento:\n")
print(f"  Costo óptimo      : {mejor_costo}   (alcanzado por: {', '.join(optimos)})")
print(f"  Más rápido        : {mas_rapido['nombre']} ({mas_rapido['tiempo_ms']:.2f} ms)")
print(f"  Menos expandidos  : {menos_exp['nombre']} ({menos_exp['expandidos']} nodos)")
print(f"  Menos memoria     : {menos_mem['nombre']} ({menos_mem['memoria_kb']:.1f} KB)")
print(f"\n  A* expandió {por['BFS']['expandidos'] / por['A*']['expandidos']:.2f}x menos nodos que BFS "
      f"({por['BFS']['expandidos']} → {por['A*']['expandidos']}) y ahorró "
      f"{por['BFS']['costo'] - por['A*']['costo']} de costo.")
print(f"  DFS resolvió en {por['DFS']['pasos']} pasos con costo {por['DFS']['costo']} "
      f"({por['DFS']['costo'] / mejor_costo:.1f}x el óptimo): rápido de ejecutar, mala solución.")

# --- Gráficos de barras ---
nombres = [r["nombre"] for r in bench]
colores = [C_AISL, C_INF, C_LIMPIO]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), facecolor=BG)

def barras(ax, valores, titulo, ylabel, etiquetas=None, err=None, linea=None):
    """Gráfico de barras con el valor escrito sobre cada barra."""
    ax.set_facecolor(BG)
    ax.bar(nombres, valores, color=colores, yerr=err, capsize=5, ecolor=BLANCO)
    ax.set_title(titulo, color=BLANCO, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel, color=TEXTO)
    ax.tick_params(colors=TEXTO)
    ax.grid(True, axis="y", color="#3A3A3A")
    ax.set_axisbelow(True)
    for s in ax.spines.values():
        s.set_visible(False)
    top = max(valores) * 0.02 + (max(err) if err else 0)
    for i, v in enumerate(valores):
        ax.text(i, v + top, etiquetas[i] if etiquetas else str(v), ha="center", color=BLANCO, fontsize=10)
    if linea is not None:
        ax.axhline(linea, color=BLANCO, linestyle="--", alpha=0.6, label=f"Óptimo = {linea}")
        ax.legend(facecolor=BG, labelcolor=BLANCO, edgecolor=BG)

barras(axes[0], [r["costo"] for r in bench], "Costo del plan", "Costo total", linea=mejor_costo)
barras(axes[1], [r["expandidos"] for r in bench], "Nodos expandidos", "Nodos")
barras(axes[2], [r["tiempo_ms"] for r in bench], "Tiempo de ejecución", "Tiempo (ms)",
       etiquetas=[f"{r['tiempo_ms']:.2f}" for r in bench], err=[r["tiempo_std"] for r in bench])
plt.tight_layout()
plt.show()